In [17]:
class Scalar:
    def __init__(self, value, _children=()):
        self.value = value
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda: None

    def __add__(self, other):
        out = Scalar(self.value + other.value, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Scalar(self.value * other.value, (self, other))

        def _backward():
            self.grad += out.grad * other.value
            other.grad += out.grad * self.value

        out._backward = _backward
        return out

    def __pow__(self, other):
        out = Scalar(self.value ** other.value, (self,))

        def _backward():
            self.grad += out.grad * other.value * self.value ** (other.value - 1)

        out._backward = _backward
        return out

    def __neg__(self):
        out = Scalar(-self.value, (self,))

        def _backward():
            self.grad -= out.grad

        out._backward = _backward
        return out

    def __sub__(self, other):
        return self + (-other)

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [19]:
import torch

# Using your Scalar class for: f(x, y) = x*y + y^3
# Given: x = 3, y = 2
x_s = Scalar(3.0)
y_s = Scalar(2.0)

out_s = (x_s * y_s) + (y_s ** Scalar(3.0))
out_s.backward()  # use the class-defined backward() method

print("Scalar results")
print(f"f(x,y) = {out_s.value}")
print(f"df/dx = {x_s.grad}")
print(f"df/dy = {y_s.grad}")

# Reference check with PyTorch
x_t = torch.tensor(3.0, requires_grad=True)
y_t = torch.tensor(2.0, requires_grad=True)
out_t = (x_t * y_t) + (y_t ** 3)
out_t.backward()

print("\nPyTorch results")
print(f"f(x,y) = {out_t.item()}")
print(f"df/dx = {x_t.grad.item()}")
print(f"df/dy = {y_t.grad.item()}")

print("\nMatch check")
print(f"x-gradient match: {abs(x_s.grad - x_t.grad.item()) < 1e-9}")
print(f"y-gradient match: {abs(y_s.grad - y_t.grad.item()) < 1e-9}")

print("\nGraph flow: x,y -> (x*y), (y^3) -> out = (x*y) + (y^3)")

Scalar results
f(x,y) = 14.0
df/dx = 2.0
df/dy = 15.0

PyTorch results
f(x,y) = 14.0
df/dx = 2.0
df/dy = 15.0

Match check
x-gradient match: True
y-gradient match: True

Graph flow: x,y -> (x*y), (y^3) -> out = (x*y) + (y^3)
